# Landmark Feature + Threshold + Ensemble Experiment

? ???? ??/?? ?? ??, ?? ??? ?? **?? ?? FaceMesh landmark**? ???? ??? ? ????? ?????.

?? ??? ? ?????.

1. **Landmark ?? ?? ??**: ?? 478? landmark ??? ??, ??, ??? ??? ?????.
2. **Threshold ??**: `0.5` ?? ?? ?? validation set?? macro-F1 ?? anxious recall ?? threshold? ????.
3. **?? ?? ???**: ?? test split?? seed? ?? ???? ??? ??? ???? ????.

??: ???? `.pth` ??? ??? ?? ?? bundle? `ensemble_config.json`? ?? ???? ?????.

In [ ]:
from pathlib import Path
import itertools
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

BASE = Path.cwd().resolve()
if BASE.name == "ML":
    BASE = BASE.parent

PYTHON = BASE / ".venv" / "Scripts" / "python.exe"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SCRIPT = BASE / "ML" / "train_landmark_feature_ensemble_2gpu_ddp.py"
VIDEO_DIR = BASE / "video"
CACHE = BASE / "ML" / "cache_landmarks_7class.npz"
OUT_DIR = BASE / "ML" / "experiments_feature_ensemble"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_BEST_ACC = 0.853053752704277
REFERENCE_BEST_F1 = 0.8527247784153322
REFERENCE_MODEL = BASE / "ML" / "experiments_ab_2gpu_tuning" / "tune_f_small_model" / "seed_123" / "direct_binary_best.pth"

print("BASE:", BASE)
print("PYTHON:", PYTHON)
print("SCRIPT:", SCRIPT)
print("REFERENCE:", REFERENCE_MODEL)

## 1. Run Feature Models

?? ??? **split_seed=123 ??**???. ? ?? ??? ?? train/val/test subject split? ?????.
?? `seed`? ??? ?? ???? ?? ??? ??? ???, ??? ?? test set?? ??????.

???? compact feature 3?? ??? ??? ?????. ??? ? ??? `RUN_OPTIONAL_WIDE=True`? wide feature? ??? ? ????.

In [ ]:
RUN_OPTIONAL_WIDE = False
RUN_FINETUNE_BRANCH = False  # True? 7-class pretrain + binary finetune?? ??? ?????. ??? ?? ?????.

COMMON = {
    "num_gpus": 2,
    "batch_size_per_gpu": 256,
    "extract_workers": 12,
    "loader_workers": 4,
    "max_per_zip": 5000,
    "split_seed": 123,
    "test_size": 0.15,
    "val_size": 0.15,
    "epochs_binary_direct": 140,
    "epochs_pretrain7": 160,
    "epochs_binary_finetune": 100,
    "lr_binary_direct": 2e-4,
    "lr_pretrain7": 8e-4,
    "lr_binary_finetune": 3e-4,
    "weight_decay": 1e-4,
    "hidden_dim": 384,
    "dropout_block": 0.30,
    "dropout_head": 0.30,
    "label_smoothing": 0.01,
    "noise_std": 0.01,
    "early_stop_patience": 35,
    "early_stop_min_delta": 1e-4,
    "threshold_recall_target": 0.85,
}

RUNS = [
    {"name": "raw_geom_compact_seed123", "seed": 123, "feature_mode": "raw_geom", "feature_preset": "compact"},
    {"name": "raw_geom_compact_seed42", "seed": 42, "feature_mode": "raw_geom", "feature_preset": "compact"},
    {"name": "raw_geom_compact_seed7", "seed": 7, "feature_mode": "raw_geom", "feature_preset": "compact"},
]

if RUN_OPTIONAL_WIDE:
    RUNS += [
        {"name": "raw_geom_wide_seed123", "seed": 123, "feature_mode": "raw_geom", "feature_preset": "wide"},
        {"name": "geom_only_compact_seed123", "seed": 123, "feature_mode": "geom", "feature_preset": "compact"},
    ]


def add_arg(cmd, name, value):
    cmd += [f"--{name.replace('_', '-')}", str(value)]


def build_cmd(run):
    out = OUT_DIR / run["name"]
    cmd = [
        str(PYTHON), str(SCRIPT),
        "--base-dir", str(VIDEO_DIR),
        "--out-dir", str(out),
        "--cache-path", str(CACHE),
        "--seed", str(run["seed"]),
        "--feature-mode", run["feature_mode"],
        "--feature-preset", run["feature_preset"],
        "--scaler-fit", "binary_train",
    ]
    for k, v in COMMON.items():
        add_arg(cmd, k, v)
    if RUN_FINETUNE_BRANCH:
        cmd.append("--train-finetune")
    cmd.append("--amp")
    return cmd, out

run_records = []
for run in RUNS:
    cmd, out = build_cmd(run)
    summary = out / "feature_ab_summary.json"
    run_records.append({**run, "out_dir": str(out), "summary": str(summary)})
    if summary.exists():
        print(f"SKIP existing: {run['name']}")
        continue
    out.mkdir(parents=True, exist_ok=True)
    print("=" * 80)
    print("RUN:", run["name"])
    print("CMD:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(BASE), check=True)

pd.DataFrame(run_records)

## 2. Single Model Results With Threshold Calibration

? ??? ? ?? test ??? ???.

- `default_0p5`: ?? ???? anxious probability >= 0.5
- `best_macro_f1`: validation set?? macro-F1? ?? ?? threshold? test? ??
- `best_macro_with_anxious_recall...`: validation?? anxious recall ??? ???? threshold ? macro-F1? ?? ?

In [ ]:
def flatten_single_results():
    rows = []
    for p in sorted(OUT_DIR.glob("*/feature_ab_summary.json")):
        with open(p, "r", encoding="utf-8") as f:
            s = json.load(f)
        exp = p.parent.name
        for model_name, result in s["results"].items():
            for threshold_name, metrics in result["test_by_val_threshold"].items():
                rows.append({
                    "exp": exp,
                    "model": model_name,
                    "threshold_policy": threshold_name,
                    "threshold": metrics["threshold"],
                    "accuracy": metrics["accuracy"],
                    "macro_f1": metrics["macro_f1"],
                    "balanced_accuracy": metrics["balanced_accuracy"],
                    "neutral_recall": metrics["neutral_recall"],
                    "anxious_recall": metrics["anxious_recall"],
                    "anxious_precision": metrics["anxious_precision"],
                    "summary": str(p),
                })
    return pd.DataFrame(rows)

single_df = flatten_single_results()
if len(single_df):
    display(single_df.sort_values(["macro_f1", "accuracy"], ascending=False).head(30))
else:
    print("?? feature_ab_summary.json ??? ????. ? ?? ?? ?? ?????.")

## 3. Probability Ensemble

???? ?? split ??? ?? `direct_binary_*_probs.npz`? ?????.
??? threshold? validation ?? ???? ?? ??, ? threshold? test ?? ??? ?????.

In [ ]:
def metrics_at_threshold(y_true, prob_anxious, threshold):
    y_pred = (prob_anxious >= threshold).astype(np.int64)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "neutral_recall": float(recall_score(y_true, y_pred, pos_label=0)),
        "anxious_recall": float(recall_score(y_true, y_pred, pos_label=1)),
        "anxious_precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }


def find_best_threshold(y_true, probs, metric="macro_f1"):
    prob_anxious = probs[:, 1]
    grid = np.unique(np.concatenate([np.linspace(0.01, 0.99, 197), prob_anxious]))
    rows = [metrics_at_threshold(y_true, prob_anxious, t) for t in grid]
    return max(rows, key=lambda r: (r[metric], r["balanced_accuracy"], r["accuracy"]))


def load_prob_pair(run_dir, stage="direct_binary"):
    run_dir = Path(run_dir)
    val = np.load(run_dir / f"{stage}_val_probs.npz")
    test = np.load(run_dir / f"{stage}_test_probs.npz")
    return {
        "name": run_dir.name,
        "run_dir": run_dir,
        "val_idx": val["indices"],
        "val_y": val["y_true"],
        "val_probs": val["probs"],
        "test_idx": test["indices"],
        "test_y": test["y_true"],
        "test_probs": test["probs"],
        "bundle": run_dir / f"{stage}_bundle.pth",
    }

models = []
for p in sorted(OUT_DIR.glob("*/direct_binary_val_probs.npz")):
    models.append(load_prob_pair(p.parent, "direct_binary"))

if len(models) < 2:
    print("?????? ?? 2? ??? ??? ??? ?????.")
else:
    base_val_idx = models[0]["val_idx"]
    base_test_idx = models[0]["test_idx"]
    base_val_y = models[0]["val_y"]
    base_test_y = models[0]["test_y"]
    for m in models[1:]:
        assert np.array_equal(base_val_idx, m["val_idx"]), f"val split mismatch: {m['name']}"
        assert np.array_equal(base_test_idx, m["test_idx"]), f"test split mismatch: {m['name']}"
        assert np.array_equal(base_val_y, m["val_y"]), f"val y mismatch: {m['name']}"
        assert np.array_equal(base_test_y, m["test_y"]), f"test y mismatch: {m['name']}"

    rows = []
    max_size = min(len(models), 5)
    for k in range(2, max_size + 1):
        for combo in itertools.combinations(models, k):
            names = [m["name"] for m in combo]
            val_probs = np.mean([m["val_probs"] for m in combo], axis=0)
            test_probs = np.mean([m["test_probs"] for m in combo], axis=0)
            val_best = find_best_threshold(base_val_y, val_probs, metric="macro_f1")
            test_metrics = metrics_at_threshold(base_test_y, test_probs[:, 1], val_best["threshold"])
            rows.append({
                "kind": "ensemble",
                "members": ", ".join(names),
                "n": k,
                "threshold_from_val": val_best["threshold"],
                **{f"test_{key}": value for key, value in test_metrics.items() if key != "confusion_matrix"},
                "confusion_matrix": test_metrics["confusion_matrix"],
                "bundle_paths": [str(m["bundle"]) for m in combo],
            })

    ensemble_df = pd.DataFrame(rows).sort_values(["test_macro_f1", "test_accuracy"], ascending=False)
    display(ensemble_df.head(20))

    best = ensemble_df.iloc[0].to_dict()
    ensemble_config = {
        "method": "probability_average",
        "class_names": ["Neutral", "Anxious"],
        "threshold": float(best["threshold_from_val"]),
        "selected_by": "validation macro-F1, evaluated on fixed binary test split",
        "members": best["members"].split(", "),
        "bundle_paths": best["bundle_paths"],
        "test_metrics": {k.replace("test_", ""): v for k, v in best.items() if k.startswith("test_")},
        "reference_previous_best": {
            "accuracy": REFERENCE_BEST_ACC,
            "macro_f1": REFERENCE_BEST_F1,
            "model": str(REFERENCE_MODEL),
        },
    }
    config_path = OUT_DIR / "best_ensemble_config.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(ensemble_config, f, ensure_ascii=False, indent=2)
    print("Saved:", config_path)

## 4. Compare Against Previous Best

?? ??? ?? ?? ?? `tune_f_small_model/seed_123/direct_binary_best.pth`? test accuracy???.

In [ ]:
plot_rows = []
if 'single_df' in globals() and len(single_df):
    tmp = single_df[single_df["threshold_policy"] == "best_macro_f1"].copy()
    tmp["label"] = tmp["exp"] + "\n" + tmp["model"]
    for _, r in tmp.iterrows():
        plot_rows.append({"label": r["label"], "accuracy": r["accuracy"], "macro_f1": r["macro_f1"], "kind": "single"})

if 'ensemble_df' in globals() and len(ensemble_df):
    for _, r in ensemble_df.head(5).iterrows():
        label = "ENS " + str(r["n"]) + " models\n" + str(r["members"])[:45]
        plot_rows.append({"label": label, "accuracy": r["test_accuracy"], "macro_f1": r["test_macro_f1"], "kind": "ensemble"})

plot_df = pd.DataFrame(plot_rows).sort_values("macro_f1", ascending=False).head(12)
if len(plot_df):
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ["tab:orange" if k == "ensemble" else "tab:blue" for k in plot_df["kind"]]
    ax.bar(range(len(plot_df)), plot_df["accuracy"], color=colors, alpha=0.85)
    ax.axhline(REFERENCE_BEST_ACC, color="red", linestyle="--", label=f"previous best acc={REFERENCE_BEST_ACC:.4f}")
    ax.set_xticks(range(len(plot_df)))
    ax.set_xticklabels(plot_df["label"], rotation=35, ha="right")
    ax.set_ylim(max(0.75, plot_df["accuracy"].min() - 0.03), min(0.92, max(plot_df["accuracy"].max(), REFERENCE_BEST_ACC) + 0.03))
    ax.set_ylabel("Test Accuracy")
    ax.set_title("Feature / Threshold / Ensemble Results")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("??? ??? ????.")

## 5. Decision Guide

- ?? ??? ?? ??? ???: ?? `*_bundle.pth`? ?? ??? ?????.
- ???? ?? ??? ???: `best_ensemble_config.json`? ? ?? bundle ?? ?? ?? ??? ????.
- ???? ?? ???: ? ??? ?????. ?static landmark??? ???? ???/?? ??? ?????? ??? ???.

In [ ]:
if 'ensemble_df' in globals() and len(ensemble_df):
    best_ens = ensemble_df.iloc[0]
    print("Best ensemble")
    print("members:", best_ens["members"])
    print("threshold:", best_ens["threshold_from_val"])
    print("accuracy:", best_ens["test_accuracy"])
    print("macro_f1:", best_ens["test_macro_f1"])
    print("anxious_recall:", best_ens["test_anxious_recall"])
    print("previous best accuracy:", REFERENCE_BEST_ACC)

if 'single_df' in globals() and len(single_df):
    best_single = single_df.sort_values(["macro_f1", "accuracy"], ascending=False).iloc[0]
    print("\nBest single")
    print(best_single[["exp", "model", "threshold_policy", "threshold", "accuracy", "macro_f1", "anxious_recall"]])